# Apache Avro - Rust

All 9 Rust examples from [docs/avro.md](https://platob.github.io/yggdryl/avro/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

## Arrow batch reads and writes

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, StringArray};
use yggdryl::generic::IORecordOptions;
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::{DataType, Url};

let field = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("venue"),
])?
.required_field("row");
let schema = field.into_arrow_schema()?;
let batch = |ids: Vec<i64>, venues: Vec<Option<&str>>| {
    RecordBatch::try_new(
        Arc::clone(&schema),
        vec![
            Arc::new(Int64Array::from(ids)),
            Arc::new(StringArray::from(venues)),
        ],
    )
};

// The name decides Avro; the methods name the write intent.
let mut handle =
    Buffer::new().with_media_type(Url::from_str("file:///trades.avro")?.media_type());
let options = handle.record_options()?;
handle.overwrite_arrow_reader(
    yggdryl::arrow::batch_reader(Arc::clone(&schema), [batch(vec![1, 2], vec![Some("XNAS"), Some("XNYS")])?]),
    &options,
)?;
handle.append_arrow_reader(
    yggdryl::arrow::batch_reader(Arc::clone(&schema), [batch(vec![3], vec![Some("XLON")])?]),
    &options,
)?;
handle.merge_arrow_reader(
    yggdryl::arrow::batch_reader(Arc::clone(&schema), [batch(vec![2, 4], vec![Some("XPAR"), None])?]),
    &options.clone().with_merge_by_names(["id"]),
)?;

let rows = handle
    .read_arrow_reader(&options)?
    .map(|batch| batch.map(|batch| batch.num_rows()))
    .sum::<Result<usize, _>>()?;
assert_eq!(rows, 4);

### Block encoding options

In [ ]:
use yggdryl::generic::RecordOptions;
use yggdryl::MimeType;

let mut options = RecordOptions::for_mime_type(&MimeType::AVRO)?;
assert_eq!(options.avro_block_codec(), Some("deflate"));
assert_eq!(options.avro_sync_marker(), None);

options.set_avro_block_codec("zstandard")?;
options.set_avro_sync_marker(Some(b"0123456789abcdef"))?;
assert_eq!(options.avro_sync_marker(), Some(b"0123456789abcdef"));

## Flexible Scalar containers and schema methods

In [ ]:
use yggdryl::io::Buffer;
use yggdryl::{Scalar, avro, json};

let schema = json::from_utf8(
    r#"{"type":"record","name":"trade","fields":[
        {"name":"symbol","type":"string"},
        {"name":"quantity","type":"long"}]}
    "#,
)?;
let rows = [
    json::from_utf8(r#"{"symbol":"AAPL","quantity":100}"#)?,
    json::from_utf8(r#"{"symbol":"MSFT","quantity":25}"#)?,
];
let mut handle = Buffer::new();
avro::write_container(&mut handle, &schema, &[("source", "docs")], &rows)?;

let decoded = avro::read_container(&handle)?;
assert_eq!(decoded.get("source"), Some("docs"));
assert_eq!(decoded.rows.len(), 2);
assert_eq!(
    decoded.rows[0].get_key_str("symbol").and_then(Scalar::as_utf8),
    Some("AAPL")
);

## Schemas, canonical form, and fingerprints

In [ ]:
use yggdryl::avro::Schema;

let schema = Schema::from_str(
    r#"{"type": "record", "name": "trade", "doc": "one fill", "fields": [
        {"name": "symbol", "type": "string"},
        {"name": "qty", "type": "long", "field-id": 2}
    ]}"#,
)?;

assert!(!schema.clone().into_canonical_form().contains("doc"));
assert_eq!(schema.fingerprint().to_le_bytes()[0], 0xF5);
let text = String::from_utf8(yggdryl::json::into_bytes(&schema.into_json())?)?;
assert!(text.contains("field-id"));

## Logical types decode as what they mean

In [ ]:
use yggdryl::generic::TimeUnit;
use yggdryl::io::Buffer;
use yggdryl::{Timezone, Scalar, avro, json};

let schema = json::from_utf8(
    r#"{"type": "record", "name": "row", "fields": [
        {"name": "day", "type": {"type": "int", "logicalType": "date"}},
        {"name": "at", "type": {"type": "long", "logicalType": "timestamp-micros"}},
        {"name": "price", "type": {"type": "bytes", "logicalType": "decimal",
                                    "precision": 10, "scale": 2}}
    ]}"#,
)?;
let row = Scalar::from_record([
    ("day", Scalar::Date32(19_782, TimeUnit::Day, Timezone::NAIVE)),
    ("at", Scalar::DateTime64(
        1_700_000_000_000_000,
        TimeUnit::Microsecond,
        Timezone::UTC,
    )),
    ("price", Scalar::D128(18_750, 2)),
])?;

let mut handle = Buffer::new();
avro::write_container(&mut handle, &schema, &[], &[row.clone()])?;
assert_eq!(avro::read_container(&handle)?.rows[0], row);

## Reading with a different schema

In [ ]:
use yggdryl::avro::Schema;
use yggdryl::io::Buffer;
use yggdryl::{Scalar, avro, json};

let writer = json::from_utf8(
    r#"{"type":"record","name":"trade","fields":[
        {"name":"symbol","type":"string"},
        {"name":"qty","type":"int"},
        {"name":"venue","type":"string"}]}"#,
)?;
let reader = Schema::from_str(
    r#"{"type":"record","name":"trade","fields":[
        {"name":"quantity","aliases":["qty"],"type":"long"},
        {"name":"note","type":"string","default":"none"}]}"#,
)?;
let row = json::from_utf8(r#"{"symbol":"AAPL","qty":100,"venue":"XNAS"}"#)?;
let mut handle = Buffer::new();
avro::write_container(&mut handle, &writer, &[], &[row])?;

let decoded = avro::read_container_resolved(&handle, &reader)?;
assert_eq!(
    decoded.rows[0].get_key_str("quantity").and_then(Scalar::as_i64),
    Some(100),
);
assert_eq!(decoded.rows[0].len(), 2, "unwanted writer fields are skipped");

## Streaming a large container

In [ ]:
use yggdryl::io::Buffer;
use yggdryl::{Scalar, avro, json};

let schema = json::from_utf8(r#"{"type":"record","name":"row","fields":[
    {"name":"id","type":"long"}]}"#)?;
let rows: Vec<Scalar> = (0..3)
    .map(|id| Scalar::from_record([("id", Scalar::from(id))]))
    .collect::<Result<_, _>>()?;
let mut handle = Buffer::new();
avro::write_container(&mut handle, &schema, &[], &rows)?;

let mut blocks = avro::read_blocks(&handle)?;
assert_eq!(blocks.schema().kind(), "record");
while let Some(block) = blocks.next_block()? {
    assert_eq!(block.rows()?.len() as u64, block.count());
}

## Single-object encoding

In [ ]:
use yggdryl::avro::Schema;
use yggdryl::{Scalar, avro};

let schema = Schema::from_str(r#"{"type":"record","name":"tick","fields":[
    {"name":"price","type":"double"}]}"#)?;
let value = Scalar::from_record([("price", Scalar::from(187.5))])?;
let framed = avro::into_single_object_vec(&schema, &value)?;

assert_eq!(&framed[..2], &[0xC3, 0x01]);
assert_eq!(avro::from_single_object_slice(&framed, &schema)?, value);

## Codecs and limits

In [ ]:
use yggdryl::io::Buffer;
use yggdryl::{Limits, Scalar, avro, json};

let schema = json::from_utf8(r#""long""#)?;
let mut bytes = Buffer::new();
avro::write_container(&mut bytes, &schema, &[], &[Scalar::I64(7)])?;

let limits = Limits::new(8, 1_024, 8, 1);
assert_eq!(avro::read_container_with_limits(&bytes, limits)?.rows, [Scalar::I64(7)]);